# Use case — `Utility/Inference.py`

Run component-level quasar inference with the Random Forest/Catch22 model and EDSM-Lite. This notebook does **not** use the lens-pair models.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

from Utility import Inference

In [ ]:
INPUT_CSV = PROJECT_ROOT / "data" / "cleaned_lightcurves.csv"  # Canonical component light curves.
OUTPUT_CSV = PROJECT_ROOT / "results" / "quasar_predictions.csv" # Component-level predictions.
MODELS_DIR = PROJECT_ROOT / "models"                              # Directory containing both RF and EDSM checkpoints.

DOMAIN = "both"                 # "real", "synthetic" or "both" trained-data domains.
BATCH_SIZE = 256                # EDSM sequences scored simultaneously; lower if memory is limited.
DEVICE = "auto"                 # "auto" selects CUDA, Apple MPS or CPU; explicit choices are also accepted.
SEED = 42                       # Reproducibility seed for NumPy, Python and PyTorch operations.
INCLUDE_PROBABILITIES = True    # Keep RFprob/EDSMprob in addition to thresholded predictions.
ADD_FINAL_PRED = True           # Add logical-OR candidate columns to reduce false negatives.

RF_MIN_MEASURES = None          # None uses the value saved with each RF; integer overrides it.
RF_MERGE_WINDOW_DAYS = None     # None uses the saved RF aggregation window; float overrides it.
EDSM_MIN_MEASURES = None        # None uses the checkpoint/default sequence minimum.
EDSM_MERGE_WINDOW_DAYS = None   # None uses the checkpoint/default EDSM aggregation window.

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Supply the cleaned canonical CSV first: {INPUT_CSV}")

In [ ]:
cli = [
    str(INPUT_CSV),
    "--domain", DOMAIN,
    "--output", str(OUTPUT_CSV),
    "--models-dir", str(MODELS_DIR),
    "--batch-size", str(BATCH_SIZE),
    "--device", DEVICE,
    "--seed", str(SEED),
]
if INCLUDE_PROBABILITIES:
    cli.append("--include-probabilities")
if ADD_FINAL_PRED:
    cli.append("--add-final-pred")
for flag, value in [
    ("--rf-min-measures", RF_MIN_MEASURES),
    ("--rf-merge-window-days", RF_MERGE_WINDOW_DAYS),
    ("--edsm-min-measures", EDSM_MIN_MEASURES),
    ("--edsm-merge-window-days", EDSM_MERGE_WINDOW_DAYS),
]:
    if value is not None:
        cli.extend([flag, str(value)])

args = Inference.build_parser().parse_args(cli)
predictions = Inference.run_inference(args)
display(predictions.head())